# Practical Assignment 03 - Retinal Disease Detection
**Classical Machine Learning and Neural Networks on ODIR-5K fundus images (Normal vs Abnormal)**

Part 1 of the notebook: environment check, dataset audit and patient-grouped split.
Model training (Parts 2-4) will be added after the audit has been reviewed.

All cells run top to bottom. The notebook calls the same code as the scripts in `scripts/`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Image as DisplayImage, display

from retina.config import load_config, resolve_paths
from retina.utils import library_versions, set_seed, setup_logging, save_json, load_json, timestamp

cfg = load_config(ROOT / "configs" / "config.yaml")
paths = resolve_paths(cfg)
log = setup_logging(paths.logs_dir / "notebook_part1.log")
set_seed(cfg["seed"])
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Project root :", ROOT)
print("Dataset root :", paths.dataset_root, "| exists:", paths.dataset_root.exists())
print("Image folder :", paths.image_dir, "| exists:", paths.image_dir.exists())

## 2. Environment

In [ ]:
versions = library_versions()
print(pd.Series(versions).to_string())
try:
    import torch
    print("\nCUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print("GPU:", props.name, "| VRAM GB:", round(props.total_memory / 1024**3, 2))
except ImportError:
    print("PyTorch not installed")

## 3. Dataset audit
Set `RERUN_AUDIT = True` to recompute; otherwise the saved report from `scripts/01_data_audit.py` is loaded.

In [ ]:
from retina.data_audit import run_audit, summary_text

RERUN_AUDIT = False
report_path = paths.audit_dir / "audit_report.json"
if RERUN_AUDIT or not report_path.exists():
    report = run_audit(cfg, paths)
    report["meta"] = {"timestamp": timestamp(), "seed": cfg["seed"], "library_versions": versions,
                      "python_executable": sys.executable, "source": "notebook"}
    save_json(report, report_path)
    (paths.audit_dir / "audit_summary.txt").write_text(summary_text(report), encoding="utf-8")
else:
    report = load_json(report_path)
print(summary_text(report))

### 3.1 Image-level labels versus patient-level flags

In [ ]:
audit = pd.read_csv(paths.audit_dir / "labeled_images_audit.csv")
example = audit[(audit["label_code"] == "N") & (audit["audit_patient_N_flag"] == 0)]
print("Normal eyes whose patient has N = 0:", len(example))
display(example[["filename", "patient_id", "eye", "label_code", "audit_patient_flags", "audit_eye_keywords"]].head(10))

### 3.2 Excluded records, unlabeled files and duplicates

In [ ]:
excluded = pd.read_csv(paths.audit_dir / "excluded_records.csv")
print("Excluded records:", len(excluded))
display(excluded[["filename", "patient_id", "label_code", "exclusion_reasons", "label_issues", "audit_eye_keywords"]])

unlabeled = pd.read_csv(paths.audit_dir / "unlabeled_image_files.csv")
print("Files in the image folder without image-level label:", len(unlabeled))
display(unlabeled["category"].value_counts().to_frame("files"))

exact = pd.read_csv(paths.audit_dir / "exact_duplicate_groups.csv")
print("Exact duplicate groups (identical files):", len(exact))
display(exact)

### 3.3 Near duplicates across patient IDs
ODIR contains the same retina under different patient IDs. Pairs with DenseNet121 descriptor cosine
similarity >= merge threshold are merged into one split group; pairs between the review and merge
thresholds are only listed. Check both figures visually (checklist item A8).

In [ ]:
near = pd.read_csv(paths.audit_dir / "near_duplicate_pairs.csv")
print(report["duplicates"]["near_duplicates"])
display(near)
for name in ("near_duplicates_merged.png", "near_duplicates_review_band.png"):
    f = paths.figures_dir / name
    if f.exists():
        display(DisplayImage(filename=str(f)))

### 3.4 Legacy DenseNet121 embeddings

In [ ]:
pd.Series({k: v for k, v in report["legacy_embeddings"].items() if k != "reencode_check"}).to_frame("value")

In [ ]:
report["legacy_embeddings"].get("reencode_check")

## 4. Patient-grouped train / validation / test split
The split is created once by `scripts/02_make_splits.py`. Here it is loaded with checksum verification.
If it does not exist yet, it is created with the same function.

In [ ]:
from retina.data_audit import CLASS_CODES
from retina.splits import load_split, make_split, save_split, split_composition, verify_split
from retina.utils import sha256_file

summary_path = paths.splits_dir / "split_summary.json"
if not summary_path.exists():
    manifest = pd.read_csv(paths.audit_dir / "clean_manifest.csv")
    s = cfg["split"]
    df_split = make_split(manifest, {k: s[k] for k in ("train", "val", "test")}, s["seed"],
                          s["group_column"], s["min_stratum_size"])
    nd = pd.read_csv(paths.audit_dir / "near_duplicate_pairs.csv")
    checks = verify_split(df_split, s["group_column"], nd[nd["merge_into_one_split_group"].astype(bool)])
    assert checks["all_passed"], checks
    save_split(df_split, paths.splits_dir, {"timestamp": timestamp(), "seed": s["seed"], "verification": checks,
                                            "source_manifest_sha256": sha256_file(paths.audit_dir / "clean_manifest.csv"),
                                            "composition": split_composition(df_split, CLASS_CODES).to_dict(orient="records")})

splits = {name: load_split(paths.splits_dir, name) for name in ("train", "val", "test")}
df_all = pd.concat([d.assign(split=n) for n, d in splits.items()], ignore_index=True)
near = pd.read_csv(paths.audit_dir / "near_duplicate_pairs.csv")
print("Verification:", verify_split(df_all, linked_pairs=near[near["merge_into_one_split_group"].astype(bool)]))
comp = split_composition(df_all, CLASS_CODES)
display(comp.round(4))

In [ ]:
fig_path = paths.figures_dir / "split_composition.png"
if fig_path.exists():
    display(DisplayImage(filename=str(fig_path)))

## 5. Next step
Model training (embeddings + Logistic Regression, MLP, DenseNet121 with and without augmentation)
will be added after the audit results are reviewed.